In [41]:
import aiohttp
import asyncio
import re
import time
from aiohttp import ClientTimeout
import pandas as pd
import numpy as np
from tqdm import tqdm
from SPARQLWrapper import SPARQLWrapper, JSON

In [42]:
SEM = asyncio.Semaphore(20) 

async def execute_sparql(session, query, timeout=30, max_retries=3):
    if not query:
        return None

    url = "https://query.wikidata.org/sparql"
    headers = {"Accept": "application/sparql-results+json"}
    data = {"query": query, "format": "json"}

    async with SEM:  # Limit concurrent requests
        for attempt in range(1, max_retries + 1):
            try:
                async with session.post(url, data=data, headers=headers, timeout=ClientTimeout(total=timeout)) as response:
                    if response.status == 200:
                        results = await response.json()
                        return extract_answers_from_response(results)
                    elif response.status == 400:  # Query malformed
                        return None

            except aiohttp.ClientError as e:
                if attempt == max_retries:
                    return []
                await asyncio.sleep(1)
            except asyncio.TimeoutError:
                if attempt == max_retries:
                    return []
                await asyncio.sleep(1)
        return []

def extract_answers_from_response(response):
    answers = []
    if 'results' in response:
        for binding in response['results']['bindings']:
            for key, sub_answer in binding.items():
                value = sub_answer.get('value')
                if re.match(r"^https?://www\.wikidata\.org/entity/Q\d+$", value):
                    answers.append(extract_wikidata_id_from_link(value))
                else:
                    answers.append(value)
    elif 'boolean' in response:
        answers.append(response['boolean'])
    return answers

def extract_wikidata_id_from_link(link):
    match = re.search(r"https?://www\.wikidata\.org/entity/(Q\d+)", link)
    return match.group(1) if match else None

In [43]:
def clean_sparql(query):
    if isinstance(query, str) and query.lower().startswith("sql"):
        query = query.split("\n", 1)[-1]  # Remove the first line if it starts with 'sql'

    return query 

def extract_sparql(text: str) -> str:
    pattern = r"```(?:[^\n]*\n)?(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()

    return text.strip()
    
def calculate_metrics(correct, predicted):
    correct_set = set(correct)
    predicted_set = set(predicted)

    em = correct_set == predicted_set
    true_positives = len(correct_set & predicted_set)  # Intersection

    precision = true_positives / len(predicted_set) if predicted_set else 0
    recall = true_positives / len(correct_set) if correct_set else 0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return {'em': em, 'f1': f1_score, 'precision': precision, 'recall': recall}

# Validation

## QaLD

In [49]:
qald_results = pd.read_json("inference_results/pat_inference_3.json")
qald_results['predicted_query'] = qald_results['predicted_query'].apply(extract_sparql)

In [50]:
metrics_list = []

async with aiohttp.ClientSession() as session:
    for index, row in tqdm(qald_results.iterrows(), total=qald_results.shape[0]):
        gold_query = row['gold_query']
        pred_query = row['predicted_query']
    
        gold_entities = await execute_sparql(session, gold_query)
        pred_entities = await execute_sparql(session, pred_query)
    
        if not gold_entities:
            continue  
    
        if pred_entities is None:
            metric = {'em': False, 'f1': 0, 'precision': 0.0, 'recall': 0, 'incorrect': True, 'empty': False}
        else:
            metric = calculate_metrics(gold_entities, pred_entities)
            metric.update({'incorrect': False, 'empty': len(pred_entities) == 0})
    
        metric.update({'id': row['id']})
        metrics_list.append(metric)

metrics_df = pd.DataFrame(metrics_list)
metrics_df[["f1", "precision", "recall", "incorrect", "empty"]].mean().round(3)*100

100%|███████████████████████████████████████| 1199/1199 [07:12<00:00,  2.77it/s]


f1           64.9
precision    64.9
recall       64.9
incorrect     0.0
empty        26.6
dtype: float64